# Spark Exercises

11 exercises, from reading a table to reading a stream — including execution plans and narrow vs wide transformations. Write your own code in the empty cell below each exercise and compare your output with the **Expected result**.

**Datasets (small on purpose to keep cost low)**

| Dataset | Rows |
|---|---|
| `samples.bakehouse.sales_transactions` | 3,333 bakery sales |
| `samples.bakehouse.sales_franchises` | 48 bakery shops |
| `abfss://inboundfiles@unityms.dfs.core.windows.net/customer/part-0000[0-1]-*.csv` | 2 CSV files with header, 75,000 customers |

**Your tables:** write everything into `one_dbx_training.default` and start every table name with your first name, e.g. `one_dbx_training.default.john_visa_sales`. Below, `<name>` means your first name.

## Exercise 1 — Read Data and Write a Filtered Table

Read `samples.bakehouse.sales_transactions`, keep only the sales paid with **visa** (`paymentMethod`), and write the result into the table `one_dbx_training.default.<name>_visa_sales`.

**Expected result:** the table has **10 columns** and **1,083 rows**.

In [0]:
# Exercise 1

## Exercise 2 — Add New Columns and Display the Result

Read your table `<name>_visa_sales` and add two new columns:
* `sale_date` — the date part of `dateTime`
* `order_size` — `LARGE` when `quantity` is 20 or more, `MEDIUM` when it is 5 or more, otherwise `SMALL`

Write the result into `one_dbx_training.default.<name>_visa_sales_enriched` and display it.

**Expected result:** **12 columns**, **1,083 rows**. Number of rows per `order_size`: LARGE = **35**, MEDIUM = **635**, SMALL = **413**.

In [0]:
# Exercise 2

## Exercise 3 — Aggregation

From `<name>_visa_sales_enriched`, create a summary per `product` with these columns:

| Column | Meaning |
|---|---|
| `product` | product name |
| `sales_count` | number of sales |
| `total_revenue` | sum of `totalPrice` |
| `avg_quantity` | average `quantity`, rounded to 2 decimals |

Sort by `total_revenue` (highest first) and write it into `one_dbx_training.default.<name>_product_summary`.

**Expected result:** **6 rows**. First row: Pearly Pies | 171 | 3,723 | 7.26

In [0]:
# Exercise 3

## Exercise 4 — Join

Join `<name>_visa_sales_enriched` with `samples.bakehouse.sales_franchises` on `franchiseID`. Show the total revenue per franchise `country`, with columns `country` and `total_revenue`, highest first.

**Expected result:** **9 rows**. US = **7,185**, Japan = **6,810**, Australia = **2,712**

In [0]:
# Exercise 4

## Exercise 5 — Join and Top N

Using the same join as Exercise 4, find the **top 3 franchises** (franchise `name`) by total revenue.

**Expected result:** Baked Bliss = **1,872**, Sweet Sinsations = **1,596**, The Crumbly Nook = **978**

In [0]:
# Exercise 5

## Exercise 6 — Spark SQL

Using **SQL** (a `%sql` cell or `spark.sql()`), query `<name>_visa_sales_enriched` and show per `sale_date`: the number of sales and the total revenue, sorted by date.

**Expected result:** **17 rows**, from 2024-05-01 to 2024-05-17.

In [0]:
# Exercise 6

## Exercise 7 — Lazy Evaluation & Execution Plans

Using `samples.bakehouse.sales_transactions`:

1. Read the table, chain `.filter(col("paymentMethod") == "visa")`, `.groupBy("product").count()`, and `.orderBy(col("count").desc())` — but **do not** call any action yet.
2. Print a message like `"No computation yet — only a plan exists"`.
3. Call `.explain(mode="formatted")` on your DataFrame to see the physical plan.
4. Finally, call `display()` to trigger execution.

**Questions (answer in a comment):**
* Which operator in the plan shows that the filter was pushed down to the scan?
* Which operator represents the shuffle caused by `groupBy`?

**Expected result:** 6 product rows. Top row: Pearly Pies | 171.

In [0]:
# Exercise 7

## Exercise 8 — Narrow vs Wide Transformations

Using `samples.bakehouse.sales_transactions`, build **two** pipelines and call `.explain()` on each (do **not** run `display()`):

**Pipeline A — Narrow only:**
```
select("product", "quantity", "totalPrice")
  → filter(quantity > 10)
  → withColumn("product_upper", upper(col("product")))
```

**Pipeline B — Wide:**
```
groupBy("product")
  → agg(sum("totalPrice").alias("total_revenue"))
  → orderBy(col("total_revenue").desc())
```

**Questions (answer in a comment):**
* Which pipeline's plan contains an `Exchange` operator? What does that mean?
* Why does Pipeline A have **no** `Exchange`?

In [0]:
# Exercise 8

## Exercise 7 — Read CSV Files and Write a Table (Batch)

Read the CSV files from `abfss://inboundfiles@unityms.dfs.core.windows.net/customer/part-0000[0-1]-*.csv` (they have a header row, let Spark detect the column types) and write them into `one_dbx_training.default.<name>_customer_batch`, replacing any existing data.

**Expected result:** **8 columns**, **75,000 rows**.

In [0]:
# Exercise 7

## Exercise 8 — Reprocess the Batch Load

Run the Exercise 8 load again, but this time **add** the data to the existing table instead of replacing it. Count the rows and explain the result.

**Expected result:** **150,000 rows** — a batch read loads all files again every time.

In [0]:
# Exercise 8

## Exercise 9 — Read a Stream

Read the same CSV files as a **stream** with Auto Loader and write them into `one_dbx_training.default.<name>_customer_stream`.
* Use the checkpoint folder `/Volumes/one_dbx_training/default/checkpoints/<name>_customer_stream`
* Use a trigger that processes all available files and then stops

Count the rows. Then run the same cell **again** and count again. Compare with Exercise 9.

**Expected result:** **75,000 rows** after the first run and still **75,000** after the second run — the stream only picks up new files.

In [0]:
# Exercise 9